# VisionTracker Remote Server — Kaggle Setup

This notebook sets up a self-hosted vision-language model as a FastAPI server for VisionTracker.

**Supports both GGUF and Safetensors formats** (auto-detected).

## Setup Instructions

1. Create a Kaggle account at https://kaggle.com
2. Create a new notebook from this file
3. Settings → Accelerator → GPU (T4/P100)
4. Settings → Internet → On
5. Add secrets (Add-ons → Secrets):
   - `NGROK_AUTHTOKEN` - Your ngrok token
   - `SERVER_API_KEY` (optional) - For API authentication
6. Run all cells

## Cell 1: Install Dependencies

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn python-multipart pyngrok pillow pydantic

# Install llama-cpp-python with CUDA support
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python

# Install transformers for safetensors support
!pip install -q transformers accelerate torch

print("✅ Dependencies installed!")

## Cell 2: Configure ngrok

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

# Get secrets
user_secrets = UserSecretsClient()

try:
    NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
    print("✅ Loaded NGROK_AUTHTOKEN from Kaggle secrets")
except:
    NGROK_TOKEN = input("Enter your ngrok authtoken: ").strip()

if not NGROK_TOKEN:
    raise ValueError("ngrok authtoken is required!")

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok configured!")

## Cell 3: Choose and Download Model

Recommended: Gemma 3 4B IT GGUF (~3.3GB, fits on T4)

In [ ]:
import os

# Choose model format: 'gguf' or 'safetensors'
MODEL_FORMAT = "gguf"  # @param ["gguf", "safetensors"]

if MODEL_FORMAT == "gguf":
    MODEL_URL = "https://huggingface.co/lmstudio-community/gemma-3-4b-it-GGUF/resolve/main/gemma-3-4b-it-Q4_K_M.gguf"
    MODEL_FILENAME = "gemma-3-4b-it-Q4_K_M.gguf"
    MODEL_PATH = f"/kaggle/working/{MODEL_FILENAME}"
else:
    MODEL_REPO = "google/gemma-3-4b-it"
    MODEL_PATH = f"/kaggle/working/{MODEL_REPO.replace('/', '_')}"

# Download GGUF if needed
if MODEL_FORMAT == "gguf" and not os.path.exists(MODEL_PATH):
    print(f"⬇️ Downloading {MODEL_FILENAME}...")
    !wget -q --show-progress {MODEL_URL} -O {MODEL_PATH}
    print("✅ Download complete!")
elif MODEL_FORMAT == "gguf":
    print(f"✅ Model already exists")

if MODEL_FORMAT == "gguf":
    print(f"📦 Model size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

## Cell 4: Download Server Code

In [ ]:
# Download the server code
!wget -q https://raw.githubusercontent.com/your-repo/visiontracker/main/remote_server/server.py -O /kaggle/working/server.py

# For safetensors, download model
if MODEL_FORMAT == "safetensors":
    from transformers import AutoModelForVision2Seq, AutoProcessor
    print("⬇️ Downloading model...")
    processor = AutoProcessor.from_pretrained(MODEL_REPO)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_REPO,
        torch_dtype="auto",
        device_map="auto"
    )
    model.save_pretrained(MODEL_PATH)
    processor.save_pretrained(MODEL_PATH)
    print("✅ Model saved!")
    del model, processor
    import torch
    torch.cuda.empty_cache()

print("✅ Server code ready!")

## Cell 5: Start Server

⚠️ Keep this notebook running!

In [ ]:
import subprocess
import threading
import time

# Create tunnel
public_url = ngrok.connect(8000, "http")

print("━" * 60)
print("🚀 SERVER IS READY!")
print("━" * 60)
print(f"📡 Public URL: {public_url}")
print(f"   Health: {public_url}/health")
print(f"   Identify: {public_url}/identify")
print("━" * 60)
print("\n💡 VisionTracker usage:")
print(f"   python main.py --remote-url {public_url}")
print("━" * 60)

# Start server
def start_server():
    cmd = ["python", "/kaggle/working/server.py", "--model-path", MODEL_PATH, "--host", "0.0.0.0", "--port", "8000"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

print("\nServer is running. Keep this notebook active.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping...")
    ngrok.disconnect(public_url)